# TinyToy：把《LLM 推理全链路》跑成代码

这个 notebook 把文档里的玩具模型 **TinyToy** 用 PyTorch 实现一遍，并逐项复算文档中的数字：

| 章节 | 复算什么 |
|---|---|
| 第 1 章 | 4 个 Q 头 / 2 个 KV 头的 GQA、分头、prefill、逐 token decode 与 KV cache 字节账 |
| 第 2 章 | 参数量、`2N = 2304`、`KV = 32 B/token` |
| 第 3 章 | PagedAttention 的 block 分配、分页前后的并发容量 |
| 第 4 章 | 算术强度 AI 与 Roofline 拐点；本机 GEMM 的 M–吞吐趋势 |
| 第 5 章 | 完整 TinyToy 前向；**prefill 与 decode 的数值等价性** |
| 第 6 章 | cached token 的三个倍数：算力 5× / 并发 3× / 带宽 1× |
| 第 7 章 | 五道准入闸门与配置死锁 |
| 第 8 章 | 四维水位、block 粒度、与 Roofline 的自洽性检查 |

**运行环境**：右上角内核选择 **Python (minimind)**。对应的 conda 环境里已有 `torch 2.6.0`：

```bash
conda activate minimind
python -c "import torch; print(torch.__version__)"
```

> 文档约定：所有绝对时间都没有意义（玩具模型一次 forward 只有 2304 FLOPs），本 notebook 只验证**公式与比例关系**。

## 0. 配置：把附录 A.1 变成一个对象

TinyToy 的关键不是“小”，而是**所有形状关系与真实模型一致**：

- `head_dim = hidden_size / num_attention_heads`（除出来的，不是配置项）
- `group_size = num_attention_heads / num_key_value_heads`（GQA）
- KV 字节 = `2(K,V) × 层数 × H_kv × head_dim × 精度字节`

为了数值验算稳定，模型内部用 `float32`；字节账仍按文档的 **BF16 = 2 B** 计算，两件事互不影响。

In [1]:
import math
import time
from dataclasses import dataclass
from typing import Dict, List

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
torch.set_printoptions(precision=4, sci_mode=False)

print("PyTorch:", torch.__version__)
print("MPS 可用:", torch.backends.mps.is_available(), "（本 notebook 用 CPU 即可）")


@dataclass(frozen=True)
class TinyToyConfig:
    """附录 A.1 的 TinyToy 配置。"""

    hidden_size: int = 8
    num_attention_heads: int = 4
    num_key_value_heads: int = 2
    num_hidden_layers: int = 2
    intermediate_size: int = 16
    vocab_size: int = 16

    block_size: int = 4
    max_num_seqs: int = 4
    max_num_batched_tokens: int = 8
    kv_pool_bytes: int = 1024
    dtype_bytes: int = 2  # BF16

    @property
    def head_dim(self) -> int:
        return self.hidden_size // self.num_attention_heads

    @property
    def group_size(self) -> int:
        return self.num_attention_heads // self.num_key_value_heads

    @property
    def kv_bytes_per_token(self) -> int:
        return (
            2
            * self.num_hidden_layers
            * self.num_key_value_heads
            * self.head_dim
            * self.dtype_bytes
        )


cfg = TinyToyConfig()
print(cfg)
print(f"head_dim = {cfg.head_dim} = {cfg.hidden_size}/{cfg.num_attention_heads}")
print(f"GQA 每组 Q 头数 = {cfg.group_size}")
print(f"KV/token = {cfg.kv_bytes_per_token} B")

PyTorch: 2.6.0
MPS 可用: True （本 notebook 用 CPU 即可）
TinyToyConfig(hidden_size=8, num_attention_heads=4, num_key_value_heads=2, num_hidden_layers=2, intermediate_size=16, vocab_size=16, block_size=4, max_num_seqs=4, max_num_batched_tokens=8, kv_pool_bytes=1024, dtype_bytes=2)
head_dim = 2 = 8/4
GQA 每组 Q 头数 = 2
KV/token = 32 B


## 1. 第 2 章：一个 token 的两张账单

先算**计算量**这张账单。

一个权重矩阵 `[K, M]` 乘一个 token `[1, K]`，每个输出元素是 `K` 次乘 + `K` 次加 = `2K` FLOPs，共有 `M` 个输出：

$$ \text{FLOPs} = 2KM = 2 \times \text{参数量} $$

把所有 transformer block 的权重参数量记作 $N$，则每 token 前向计算量就是 $2N$。

下面用真实的 `nn.Linear` 模块统计，避免手抄错行。

In [2]:
# 每层 7 个线性权重的形状（in, out）。W_K / W_V 的 out 只有 H_kv * d = 4
layer_specs = {
    "W_Q":    (cfg.hidden_size, cfg.num_attention_heads * cfg.head_dim),
    "W_K":    (cfg.hidden_size, cfg.num_key_value_heads * cfg.head_dim),
    "W_V":    (cfg.hidden_size, cfg.num_key_value_heads * cfg.head_dim),
    "W_O":    (cfg.num_attention_heads * cfg.head_dim, cfg.hidden_size),
    "W_gate": (cfg.hidden_size, cfg.intermediate_size),
    "W_up":   (cfg.hidden_size, cfg.intermediate_size),
    "W_down": (cfg.intermediate_size, cfg.hidden_size),
}

probe = nn.ModuleDict(
    {name: nn.Linear(din, dout, bias=False) for name, (din, dout) in layer_specs.items()}
)

print(f"{'权重':<8}{'形状':<12}{'参数量':>8}{'每 token FLOPs':>16}")
print("-" * 46)
per_layer = 0
for name, (din, dout) in layer_specs.items():
    params = probe[name].weight.numel()
    per_layer += params
    print(f"{name:<8}[{din}, {dout}]".ljust(20) + f"{params:>8}{2 * params:>16}")

N = per_layer * cfg.num_hidden_layers
two_N = 2 * N
weight_bytes = N * cfg.dtype_bytes
mlp_params = 3 * cfg.hidden_size * cfg.intermediate_size

print("-" * 46)
print(f"每层参数 {per_layer}，每层 FLOPs {2 * per_layer}")
print(f"N = {per_layer} × {cfg.num_hidden_layers} = {N}")
print(f"2N = {two_N} FLOPs/token")
print(f"权重字节 = {N} × {cfg.dtype_bytes} B = {weight_bytes} B")
print(f"MLP 占计算 = {mlp_params}/{per_layer} = {mlp_params / per_layer:.1%}")
print(f"MLP 占 KV    = 0/{cfg.kv_bytes_per_token} = 0.0%")

# 用 nn.Linear 的真实参数再核对一次
assert per_layer == 576
assert N == 1152
assert two_N == 2304
assert weight_bytes == 2304
assert abs(mlp_params / per_layer - 2 / 3) < 1e-9

# 第二张账单：KV 存储
capacity = cfg.kv_pool_bytes // cfg.kv_bytes_per_token
block_bytes = cfg.block_size * cfg.kv_bytes_per_token
num_blocks = cfg.kv_pool_bytes // block_bytes

print()
print(f"KV 五因子 = 2 × {cfg.num_hidden_layers} × {cfg.num_key_value_heads} × {cfg.head_dim} × {cfg.dtype_bytes} = {cfg.kv_bytes_per_token} B/token")
print(f"池 {cfg.kv_pool_bytes} B 可装 {capacity} token")
print(f"block = {cfg.block_size} token × {cfg.kv_bytes_per_token} B = {block_bytes} B，共 {num_blocks} 个")

assert cfg.kv_bytes_per_token == 32
assert capacity == 32 and block_bytes == 128 and num_blocks == 8
assert num_blocks * cfg.block_size == capacity

权重      形状               参数量   每 token FLOPs
----------------------------------------------
W_Q     [8, 8]            64             128
W_K     [8, 4]            32              64
W_V     [8, 4]            32              64
W_O     [8, 8]            64             128
W_gate  [8, 16]          128             256
W_up    [8, 16]          128             256
W_down  [16, 8]          128             256
----------------------------------------------
每层参数 576，每层 FLOPs 1152
N = 576 × 2 = 1152
2N = 2304 FLOPs/token
权重字节 = 1152 × 2 B = 2304 B
MLP 占计算 = 384/576 = 66.7%
MLP 占 KV    = 0/32 = 0.0%

KV 五因子 = 2 × 2 × 2 × 2 × 2 = 32 B/token
池 1024 B 可装 32 token
block = 4 token × 32 B = 128 B，共 8 个


## 2. 第 1 章：手算一个 token 的一生

这是全文的核心例子。输入隐藏状态：

$$ h = [1, 0, 1, 1, 0, 1, 0, 1] $$

三个投影矩阵都是**稀疏的 0/1 矩阵**，所以结果可以完全手算：

- `W_Q [8,8]`：`q = [1, 0, 0, 1, 1, 1, 1, -1]`
- `W_K [8,4]`：`k = [1, 1, 0, 1]`
- `W_V [8,4]`：`v = [1, 1, 1, 1]`

`q` 再 reshape 成 `[4, 2]`，得到 4 个各不相同的 2 维查询向量。

In [3]:
h = torch.tensor([1.0, 0.0, 1.0, 1.0, 0.0, 1.0, 0.0, 1.0])

# 行为输入维 i，列为输出维 j（与文档 A.4 一致）
W_Q = torch.tensor(
    [
        [1, 0, 0, 0, 0, 0, 0, -1],
        [0, 1, 1, 0, 0, 0, 0, 0],
        [0, 0, 0, 1, 0, 0, 0, 0],
        [0, 0, 0, 0, 1, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 1, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 1, 0],
    ],
    dtype=torch.float32,
)

W_K = torch.zeros(8, 4)
for i, j in [(0, 0), (2, 1), (4, 2), (5, 3)]:
    W_K[i, j] = 1.0

W_V = torch.zeros(8, 4)
for i, j in [(0, 0), (2, 1), (3, 2), (5, 3)]:
    W_V[i, j] = 1.0

q = h @ W_Q
k = h @ W_K
v = h @ W_V

q_heads = q.reshape(cfg.num_attention_heads, cfg.head_dim)
k_heads = k.reshape(cfg.num_key_value_heads, cfg.head_dim)
v_heads = v.reshape(cfg.num_key_value_heads, cfg.head_dim)

print("q =", q.tolist())
print("q reshape[4,2] =", q_heads.tolist())
print("k =", k.tolist(), "->", k_heads.tolist())
print("v =", v.tolist(), "->", v_heads.tolist())

qkv_params = W_Q.numel() + W_K.numel() + W_V.numel()
print()
print(f"投影 FLOPs = 2 × {qkv_params} = {2 * qkv_params}")
print(f"权重字节   = {qkv_params} × {cfg.dtype_bytes} B = {qkv_params * cfg.dtype_bytes} B")

assert torch.equal(q, torch.tensor([1.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, -1.0]))
assert torch.equal(k, torch.tensor([1.0, 1.0, 0.0, 1.0]))
assert torch.equal(v, torch.ones(4))
assert 2 * qkv_params == 256

q = [1.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, -1.0]
q reshape[4,2] = [[1.0, 0.0], [0.0, 1.0], [1.0, 1.0], [1.0, -1.0]]
k = [1.0, 1.0, 0.0, 1.0] -> [[1.0, 1.0], [0.0, 1.0]]
v = [1.0, 1.0, 1.0, 1.0] -> [[1.0, 1.0], [1.0, 1.0]]

投影 FLOPs = 2 × 128 = 256
权重字节   = 128 × 2 B = 256 B


## 3. 第 1 章：GQA 分组注意力

KV cache 里已有历史 token **A、B**，刚写入 **C**（`L = 3`）：

```
KV head 0:  K = [[1,0],[0,1],[1,1]]   V = [[1,0],[0,1],[1,1]]
KV head 1:  K = [[1,1],[1,-1],[0,1]]  V = [[2,0],[0,2],[1,1]]
```

分组规则：`q0, q1 → KV head 0`，`q2, q3 → KV head 1`。

**关键点**：同一份 K/V、不同的 `q`，会得到完全不同的注意力分布——所以共享 K/V 不会让 4 种注意力塌成 2 种。

In [4]:
# 形状：[KV head, 历史 token, head_dim]
K_cache = torch.tensor(
    [
        [[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]],   # KV head 0: A, B, C
        [[1.0, 1.0], [1.0, -1.0], [0.0, 1.0]],  # KV head 1: A, B, C
    ]
)
V_cache = torch.tensor(
    [
        [[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]],
        [[2.0, 0.0], [0.0, 2.0], [1.0, 1.0]],
    ]
)


def gqa_attention(q_heads: torch.Tensor, K: torch.Tensor, V: torch.Tensor, scale: float = 1.0):
    """逐头做 softmax(q·Kᵀ)·V，返回输出和每一步的中间量。"""
    outs = []
    rows = []
    for hq in range(cfg.num_attention_heads):
        kv_head = hq // cfg.group_size
        scores = (K[kv_head] @ q_heads[hq]) * scale
        weights = torch.softmax(scores, dim=-1)
        out = weights @ V[kv_head]
        outs.append(out)
        rows.append((hq, kv_head, scores, weights, out))
    return torch.stack(outs), rows


outs, rows = gqa_attention(q_heads, K_cache, V_cache)  # 文档约定：省略 1/sqrt(d)

print(f"{'Q头':<5}{'KV头':<6}{'scores':<18}{'softmax 权重':<30}{'输出'}")
print("-" * 78)
for hq, kv_head, scores, weights, out in rows:
    w = [round(x, 4) for x in weights.tolist()]
    o = [round(x, 3) for x in out.tolist()]
    print(f"{hq:<5}{kv_head:<6}{str([round(x, 3) for x in scores.tolist()]):<18}{str(w):<30}{o}")

expected = torch.tensor(
    [[0.845, 0.578], [0.578, 0.845], [1.575, 0.425], [0.270, 1.730]]
)
assert torch.allclose(outs, expected, atol=1e-3)

# 共享 K/V 不塌缩：4 个输出互不相同
for i in range(len(outs)):
    for j in range(i + 1, len(outs)):
        assert not torch.allclose(outs[i], outs[j])

# out0 与 out1 正好对称
assert torch.allclose(outs[0], outs[1].flip(0))

concat = outs.reshape(-1)
print()
print("concat =", [round(x, 3) for x in concat.tolist()])
print("长度 =", concat.numel(), "= hidden_size ✓")

# attention FLOPs 校验：4 个头 × (QKᵀ: 2·L·d + AV: 2·L·d)
L = K_cache.shape[1]
attn_flops = cfg.num_attention_heads * (2 * L * cfg.head_dim + 2 * L * cfg.head_dim)
formula = 4 * L * cfg.head_dim * cfg.num_attention_heads
print(f"attention FLOPs = {attn_flops}；公式 4·L·d·H_q = {formula}")
assert attn_flops == formula == 96

Q头   KV头   scores            softmax 权重                    输出
------------------------------------------------------------------------------
0    0     [1.0, 0.0, 1.0]   [0.4223, 0.1554, 0.4223]      [0.845, 0.578]
1    0     [0.0, 1.0, 1.0]   [0.1554, 0.4223, 0.4223]      [0.578, 0.845]
2    1     [2.0, 0.0, 1.0]   [0.6652, 0.09, 0.2447]        [1.575, 0.425]
3    1     [0.0, 2.0, -1.0]  [0.1142, 0.8438, 0.042]       [0.27, 1.73]

concat = [0.845, 0.578, 0.578, 0.845, 1.575, 0.425, 0.27, 1.73]
长度 = 8 = hidden_size ✓
attention FLOPs = 96；公式 4·L·d·H_q = 96


### 3.1 补上 `1/√d` 缩放

文档手算时刻意省略了缩放（TinyToy 的 `1/√2 = 0.7071`），因为省略后分布更尖、但结论不变。

真实的 attention kernel 一定带缩放。下面用同一个函数跑一次缩放版本，观察两点：

1. 权重变得更平缓；
2. **四个头依然互不相同**——结论不依赖缩放。

In [5]:
scale = 1.0 / math.sqrt(cfg.head_dim)
outs_scaled, rows_scaled = gqa_attention(q_heads, K_cache, V_cache, scale=scale)

print(f"1/sqrt(d) = 1/sqrt({cfg.head_dim}) = {scale:.4f}\n")
print(f"{'Q头':<5}{'softmax（带缩放）':<40}{'输出（带缩放）'}")
print("-" * 78)
for hq, kv_head, scores, weights, out in rows_scaled:
    w = [round(x, 4) for x in weights.tolist()]
    o = [round(x, 3) for x in out.tolist()]
    print(f"{hq:<5}{str(w):<40}{o}")

for i in range(len(outs_scaled)):
    for j in range(i + 1, len(outs_scaled)):
        assert not torch.allclose(outs_scaled[i], outs_scaled[j])

print("\n四个输出仍互不相同 ✓；缩放只改变分布的锐度，不改变 GQA 的表达能力。")

1/sqrt(d) = 1/sqrt(2) = 0.7071

Q头   softmax（带缩放）                            输出（带缩放）
------------------------------------------------------------------------------
0    [0.4011, 0.1978, 0.4011]                [0.802, 0.599]
1    [0.1978, 0.4011, 0.4011]                [0.599, 0.802]
2    [0.576, 0.14, 0.284]                    [1.436, 0.564]
3    [0.1784, 0.7337, 0.0879]                [0.445, 1.555]

四个输出仍互不相同 ✓；缩放只改变分布的锐度，不改变 GQA 的表达能力。


### 3.2 从「一个 token 的手算」到「真的吐出下一个 token」

前三节回答的是「**一个** token 的 q / k / v 长什么样」。真实推理还要回答三个问题：

1. **分头**：`q [8]` 怎么变成 4 个 head？为什么 `k / v [4]` 只有 2 个 head？
2. **开始**：第 1 个 token 进模型时 KV cache 是空的，它凭什么能输出东西？
3. **继续**：第 2、3、… 个 token 进来时，为什么不用把前面所有 token 重算一遍？

下面用**单层、无残差、无 RoPE、无 MLP** 的最小结构把这三件事走完。所有形状与第 5 章的完整模型一致，只是砍掉了别的部件，好让每一步的数都能打印出来核对。

```text
tokens ──embed──▶ h[1,8]
                    │
        ┌───────────┼─────────────┐
        ▼           ▼             ▼
     q[4,1,2]    k[2,1,2]      v[2,1,2]
        │           │             │
        │           └── append ──▶ KV cache（每层一份 [2, L, 2]）
        ▼                         │
        │           ┌─────────────┘
        ▼           ▼
  scores[4,1,L] = q·Kᵀ/√d ──softmax──▶ w[4,1,L]
                                          │
                          out[4,1,2] = w·V（每个 Q head 一个）
                                          │
                          merge ──▶ x[1,8] ──lm_head──▶ logits[16]
                                                           │
                                                    argmax ┘→ 下一个 token
```

> **q 不进 cache。** 下一个 token 有自己的 q；历史 token 的 q 再也不会被用到。能跨 step 复用的只有历史 K、V。


### 3.2.1 分头：8 个数怎么切成 4 个 Q head / 2 个 KV head

| 张量 | 投影后（扁平） | 分头后 | 每头维度 |
|---|---|---|---|
| `q` | `[8] = H_q × d = 4 × 2` | `[4, 2]` | 2 |
| `k` | `[4] = H_kv × d = 2 × 2` | `[2, 2]` | 2 |
| `v` | `[4]` | `[2, 2]` | 2 |

分头只有两步：**`view(T, H, d)` 按 `d` 切断，再 `transpose(0, 1)` 把 head 维挪到最前**。之后所有张量都是 head-major：

```text
q: [4, T, 2]        K/V cache: [2, L, 2]        scores/weights: [4, T, L]
```

换成 head-major 是为了让 `einsum("hqd,hld->hql")` 里的 `h` 直接当批维 —— 这样注意力那几行里一个 `for hq` 都不用写。

最后一段是**反面教材**：同样 8 个数，`view(T, H, d).transpose(0, 1)` 和直接把 `view(T, H, d)` 当成 `[H, T, d]`，切出来的 head 完全不同。这是手写 attention 最常踩的坑。


In [6]:
H, HQ, HKV, D = cfg.hidden_size, cfg.num_attention_heads, cfg.num_key_value_heads, cfg.head_dim


def split_heads(x: torch.Tensor, n_heads: int) -> torch.Tensor:
    """[T, n_heads*d] -> [n_heads, T, d]：先按 d 切断，再把 head 维挪到最前。"""
    T = x.shape[0]
    return x.view(T, n_heads, D).transpose(0, 1)


def merge_heads(x: torch.Tensor) -> torch.Tensor:
    """[n_heads, T, d] -> [T, n_heads*d]：必须先 transpose 再 reshape。"""
    return x.transpose(0, 1).reshape(x.shape[1], -1)


q_split = split_heads(q[None], HQ)[:, 0]      # [4, 2]
k_split = split_heads(k[None], HKV)[:, 0]     # [2, 2]
v_split = split_heads(v[None], HKV)[:, 0]     # [2, 2]

print("第 2 节的 q 扁平向量:", q.tolist())
print("切成 4 个 Q head    :", q_split.tolist(),
      "（与第 2 节 q_heads 相同:", torch.equal(q_split, q_heads), "）")
print("切成 2 个 KV head   : k =", k_split.tolist(), " v =", v_split.tolist())

print(f"\n分组（group_size = {cfg.group_size}）：一个 KV head 被 {cfg.group_size} 个 Q head 共用")
for hq in range(HQ):
    print(f"  q{hq} {q_split[hq].tolist()}  ->  kv{hq // cfg.group_size}")

K_bc = k_split[:, None].repeat_interleave(cfg.group_size, dim=0)   # [4, 1, 2]
print("\n把 KV head 复制成 4 份，才能和 4 个 Q head 逐头点积：")
for hq in range(HQ):
    print(f"  head{hq} 用 kv{hq // cfg.group_size} 的 k = {K_bc[hq, 0].tolist()}")

print("\n反面教材：同样是 8 个数，切法不同，含义完全不同")
flat = torch.tensor([[1.0, 2.0, 3.0, 4.0], [5.0, 6.0, 7.0, 8.0]])  # 2 个 token，各 4 维 = 2 head × 2 维
right = flat.view(2, 2, 2).transpose(0, 1)   # 正确：[T, H, d] -> [H, T, d]
wrong = flat.view(2, 2, 2)                   # 错误：把 [T, H, d] 直接当成 [H, T, d]
print("  正确 head0 =", right[0].tolist(), "（token0 前两维 + token1 前两维）")
print("  错误 head0 =", wrong[0].tolist(), "（token0 前两维 + token0 后两维）")
assert torch.equal(right[0], torch.tensor([[1.0, 2.0], [5.0, 6.0]]))
assert not torch.equal(right, wrong)

第 2 节的 q 扁平向量: [1.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, -1.0]
切成 4 个 Q head    : [[1.0, 0.0], [0.0, 1.0], [1.0, 1.0], [1.0, -1.0]] （与第 2 节 q_heads 相同: True ）
切成 2 个 KV head   : k = [[1.0, 1.0], [0.0, 1.0]]  v = [[1.0, 1.0], [1.0, 1.0]]

分组（group_size = 2）：一个 KV head 被 2 个 Q head 共用
  q0 [1.0, 0.0]  ->  kv0
  q1 [0.0, 1.0]  ->  kv0
  q2 [1.0, 1.0]  ->  kv1
  q3 [1.0, -1.0]  ->  kv1

把 KV head 复制成 4 份，才能和 4 个 Q head 逐头点积：
  head0 用 kv0 的 k = [1.0, 1.0]
  head1 用 kv0 的 k = [1.0, 1.0]
  head2 用 kv1 的 k = [0.0, 1.0]
  head3 用 kv1 的 k = [0.0, 1.0]

反面教材：同样是 8 个数，切法不同，含义完全不同
  正确 head0 = [[1.0, 2.0], [5.0, 6.0]] （token0 前两维 + token1 前两维）
  错误 head0 = [[1.0, 2.0], [3.0, 4.0]] （token0 前两维 + token0 后两维）


### 3.2.2 KV cache 的物理结构：两块 buffer + 一个长度

缓存里每层只有一对 buffer：

```text
k[layer]: [H_kv, capacity, d]     ← 只在 start:start+T 这一小段写入
v[layer]: [H_kv, capacity, d]     ← 读的时候只取前 L 个
lengths[layer]: int               ← L，唯一的"已用长度"
```

三个必须记住的点：

1. **容量一开始就按 `capacity` 分配好**，`L` 只增不减地往右走。所以显存是按「最大并发 × 最大长度」预留的 —— 这正是第 4 节要解决的问题。
2. **只存 K、V**，不存 Q，也不存 h。
3. 真实实现里 **K 存的是 RoPE 之后的**（见第 5 章），因为 `q·k` 需要位置信息；本节先不引入 RoPE。


In [7]:
class ContiguousKVCache:
    """连续布局的 KV cache：每层一对 buffer，写入只追加，读出只取前 L 个。"""

    def __init__(self, cfg: TinyToyConfig, capacity: int, num_layers: int = 1):
        self.cfg, self.capacity = cfg, capacity
        self.lengths = [0] * num_layers
        shape = (cfg.num_key_value_heads, capacity, cfg.head_dim)     # [H_kv, cap, d]
        self.k = [torch.zeros(shape) for _ in range(num_layers)]
        self.v = [torch.zeros(shape) for _ in range(num_layers)]

    def append(self, layer: int, k: torch.Tensor, v: torch.Tensor) -> int:
        """k, v: [H_kv, T, d]。返回写入前的长度，也就是本批第一个 token 的位置。"""
        T = k.shape[1]
        start = self.lengths[layer]
        assert start + T <= self.capacity, "cache 满了"
        self.k[layer][:, start:start + T] = k
        self.v[layer][:, start:start + T] = v
        self.lengths[layer] = start + T
        return start

    def keys(self, layer: int) -> torch.Tensor:      # [H_kv, L, d]
        return self.k[layer][:, : self.lengths[layer]]

    def values(self, layer: int) -> torch.Tensor:    # [H_kv, L, d]
        return self.v[layer][:, : self.lengths[layer]]

    def nbytes(self, layer: int = 0, dtype_bytes: int = 2) -> int:
        """已用 token 的账：2(K,V) × H_kv × d × L × 字节。"""
        return 2 * self.cfg.num_key_value_heads * self.cfg.head_dim * self.lengths[layer] * dtype_bytes

    def reserved_bytes(self, dtype_bytes: int = 2) -> int:
        """已分配的物理字节：按 capacity 预留，与当前 L 无关。"""
        return (2 * len(self.k) * self.cfg.num_key_value_heads
                * self.capacity * self.cfg.head_dim * dtype_bytes)


kv0 = ContiguousKVCache(cfg, capacity=8, num_layers=1)
print("每层一对 buffer，形状 =", tuple(kv0.k[0].shape), "= [H_kv=2, capacity=8, d=2]")
print("初始 L =", kv0.lengths[0], "；物理预留 =", kv0.reserved_bytes(), "B（1 层 BF16）；已用 =", kv0.nbytes(), "B")
assert kv0.reserved_bytes() == 8 * 2 * HKV * D * 2

每层一对 buffer，形状 = (2, 8, 2) = [H_kv=2, capacity=8, d=2]
初始 L = 0 ；物理预留 = 128 B（1 层 BF16）；已用 = 0 B


### 3.2.3 一个能跑的最小 GQA decoder

三件事需要交代：

- **embedding 用 ±1 的手写表**：这样 h、K、V 全是整数，每一步都能拿笔核对。代价是只有 `16 / 256` 的 ±1 向量能让 4 个 q head 互不相同（`W_Q` 太稀疏了），代码里把这个筛选过程直接写了出来。
- **投影矩阵沿用第 2 节的 `W_Q / W_K / W_V`**：`nn.Linear` 的 `weight` 是 `[out, in]`，所以要转置后拷进去。
- **`lm_head` 是未训练的随机矩阵**：token 没有语义，只保证 argmax 有确定结果。真实模型里这个矩阵是训练出来的。

`forward` 里的顺序就是推理的顺序：**投影 → 分头 → 写 cache → 读 cache → 打分 → mask → softmax → 加权 → 拼头 → lm_head**。


In [8]:
import itertools

# 词表 16 个 token 的 embedding：全部用 ±1 的手写行，K/V 才会是整数。
# 额外要求：每个 token 的 4 个 q head 互不相同，否则这一层看不出 4 个头在做什么。
candidates = []
for bits in itertools.product((-1.0, 1.0), repeat=cfg.hidden_size):
    heads = {tuple(x) for x in (torch.tensor(bits) @ W_Q).view(HQ, D).tolist()}
    if len(heads) == HQ:
        candidates.append(bits)

EMB = torch.tensor([h.tolist()] + candidates[: cfg.vocab_size - 1], dtype=torch.float32)
print(f"2^{cfg.hidden_size} = {2 ** cfg.hidden_size} 个 ±1 向量里，只有 {len(candidates)} 个能让 4 个 q head 互不相同")
print(f"词表 {cfg.vocab_size} 个 token：token 0 直接用第 2 节的 h，其余取上面筛出的行")
print("EMB[0] =", EMB[0].tolist(), "（= 第 2 节的 h）")
print("EMB[1] =", EMB[1].tolist())
assert EMB.shape == (cfg.vocab_size, cfg.hidden_size)
assert len({tuple(r) for r in EMB.tolist()}) == cfg.vocab_size


class TinyGQADecoder(nn.Module):
    """单层、无残差、无 RoPE、无 MLP 的 GQA decoder：只为把数据流看清楚。"""

    def __init__(self, cfg: TinyToyConfig, table: torch.Tensor, lm_seed: int = 1234):
        super().__init__()
        self.cfg = cfg
        self.scale = 1.0 / math.sqrt(cfg.head_dim)
        self.emb = nn.Embedding(cfg.vocab_size, cfg.hidden_size)
        # 沿用第 2 节的手写矩阵；nn.Linear 的 weight 是 [out, in]，所以要转置
        self.q_proj = nn.Linear(cfg.hidden_size, cfg.num_attention_heads * cfg.head_dim, bias=False)
        self.k_proj = nn.Linear(cfg.hidden_size, cfg.num_key_value_heads * cfg.head_dim, bias=False)
        self.v_proj = nn.Linear(cfg.hidden_size, cfg.num_key_value_heads * cfg.head_dim, bias=False)
        # 未训练的 lm_head：只为让 argmax 有确定结果（整体缩放不改变 argmax）
        self.lm_head = nn.Linear(cfg.hidden_size, cfg.vocab_size, bias=False)
        with torch.no_grad():
            self.emb.weight.copy_(table)
            self.q_proj.weight.copy_(W_Q.T)
            self.k_proj.weight.copy_(W_K.T)
            self.v_proj.weight.copy_(W_V.T)
            self.lm_head.weight.copy_(
                torch.randn(cfg.vocab_size, cfg.hidden_size,
                            generator=torch.Generator().manual_seed(lm_seed)) * 0.15
            )

    def forward(self, tokens: torch.Tensor, cache: ContiguousKVCache,
                layer: int = 0, trace: bool = False) -> dict:
        cfg = self.cfg
        T = tokens.shape[0]
        x = self.emb(tokens)                                              # [T, 8]
        q = split_heads(self.q_proj(x), cfg.num_attention_heads)          # [4, T, 2]
        k = split_heads(self.k_proj(x), cfg.num_key_value_heads)          # [2, T, 2]
        v = split_heads(self.v_proj(x), cfg.num_key_value_heads)          # [2, T, 2]

        L0 = cache.append(layer, k, v)        # 先写再读：本 token 也能看到自己
        L = L0 + T
        K, V = cache.keys(layer), cache.values(layer)                     # [2, L, 2]
        Kb = K.repeat_interleave(cfg.group_size, dim=0)                   # [4, L, 2]
        Vb = V.repeat_interleave(cfg.group_size, dim=0)

        scores = torch.einsum("hqd,hld->hql", q, Kb) * self.scale         # [4, T, L]
        q_pos, k_pos = torch.arange(L0, L), torch.arange(L)
        causal = k_pos[None, :] <= q_pos[:, None]                         # [T, L]
        scores = scores.masked_fill(~causal[None], float("-inf"))
        weights = torch.softmax(scores, dim=-1)
        out = torch.einsum("hql,hld->hqd", weights, Vb)                   # [4, T, 2]

        merged = merge_heads(out)                                         # [T, 8]
        result = {"logits": self.lm_head(merged)}                         # [T, 16]
        if trace:
            result.update(x=x, q=q, k=k, v=v, K=Kb, V=Vb, causal=causal,
                          scores=scores, weights=weights, out=out, merged=merged, L0=L0)
        return result


dec = TinyGQADecoder(cfg, EMB)
print("\nnn.Linear 里存的是转置后的矩阵，等价性检查:", torch.allclose(dec.q_proj(h[None]), (h @ W_Q)[None]))

2^8 = 256 个 ±1 向量里，只有 16 个能让 4 个 q head 互不相同
词表 16 个 token：token 0 直接用第 2 节的 h，其余取上面筛出的行
EMB[0] = [1.0, 0.0, 1.0, 1.0, 0.0, 1.0, 0.0, 1.0] （= 第 2 节的 h）
EMB[1] = [-1.0, -1.0, 1.0, 1.0, -1.0, -1.0, -1.0, 1.0]

nn.Linear 里存的是转置后的矩阵，等价性检查: True


### 3.2.4 prefill：一次算完 prompt，并吐出第 1 个 token

prompt 有 3 个 token，一次性前向。注意四个细节：

1. **causal mask 是下三角**：第 i 个 token 只能看 `0..i`。
2. **只取最后一个位置的 logits** 去生成；前面位置的 logits 在推理时用不到（训练时才算全部）。
3. **cache 的 buffer 形状始终不变**，只是前 `L` 个位置被填上了 —— 这就是「按容量预留」的样子。
4. **单个 token 的注意力必然是 1**：`L = 1` 时 `softmax([s]) = [1]`，输出恒等于 `v`，与 `q` 无关。所以「共享 K/V 不塌缩」在单 token 上根本演示不出来，必须攒到 ≥2 个 token 才看得见。


In [9]:
prompt = torch.tensor([0, 1, 2])
cache = ContiguousKVCache(cfg, capacity=8, num_layers=1)

print("写入前：buffer 形状 =", tuple(cache.k[0].shape), " L =", cache.lengths[0])
with torch.no_grad():
    pre = dec(prompt, cache, trace=True)
print("写入后：L =", cache.lengths[0], "，buffer 形状不变 —— 只是前 L 个位置被填了")

print("\n1) 三个 token 的 h（= embedding 查表）")
for t in range(len(prompt)):
    print(f"   pos {t}  token {int(prompt[t]):>2}  h = {[int(x) for x in pre['x'][t].tolist()]}")

print("\n2) 写进 cache 的 K / V，形状 [H_kv=2, L=3, d=2]")
print("   K =", cache.keys(0).to(torch.int).tolist())
print("   V =", cache.values(0).to(torch.int).tolist())

print("\n3) causal mask（第 i 行 = 第 i 个 token 能看到哪些位置）")
for i, row in enumerate(pre["causal"].tolist()):
    print(f"   pos {i}: {[int(x) for x in row]}")

print("\n4) 最后一个 token（pos 2）在 4 个 Q head 上的注意力")
for hq in range(HQ):
    s = [round(x, 3) for x in pre["scores"][hq, -1].tolist()]
    w = [round(x, 3) for x in pre["weights"][hq, -1].tolist()]
    o = [round(x, 3) for x in pre["out"][hq, -1].tolist()]
    print(f"   q{hq}(kv{hq // cfg.group_size})  scores={s}  weights={w}  out={o}")

print("\n5) 拼回 8 维 → lm_head → logits → argmax")
logits = pre["logits"][-1]
top2 = torch.topk(logits, 2)
print("   merged =", [round(x, 3) for x in pre["merged"][-1].tolist()])
print("   logits =", [round(x, 3) for x in logits.tolist()])
print(f"   top2: token {top2.indices[0].item()} ({top2.values[0]:.4f}) / "
      f"token {top2.indices[1].item()} ({top2.values[1]:.4f})")
first_next = int(top2.indices[0])
print("   下一个 token =", first_next)

print("\n6) 只有 1 个 token 时（L = 1）：softmax([s]) 恒等于 [1]，输出就是 v")
cache1 = ContiguousKVCache(cfg, capacity=8, num_layers=1)
with torch.no_grad():
    one = dec(torch.tensor([first_next]), cache1, trace=True)
print("   weights =", one["weights"][:, 0].tolist())
print("   out     =", one["out"][:, 0].tolist())
print("   v       =", one["v"][:, 0].tolist())
assert torch.allclose(one["weights"], torch.ones_like(one["weights"]))
for hq in range(HQ):
    assert torch.allclose(one["out"][hq, 0], one["v"][hq // cfg.group_size, 0])
print("   → out 恒等于 v：单 token 时 q 完全没参与，4 个 head 也只有 2 种输出")

写入前：buffer 形状 = (2, 8, 2)  L = 0
写入后：L = 3 ，buffer 形状不变 —— 只是前 L 个位置被填了

1) 三个 token 的 h（= embedding 查表）
   pos 0  token  0  h = [1, 0, 1, 1, 0, 1, 0, 1]
   pos 1  token  1  h = [-1, -1, 1, 1, -1, -1, -1, 1]
   pos 2  token  2  h = [-1, -1, 1, 1, -1, -1, 1, 1]

2) 写进 cache 的 K / V，形状 [H_kv=2, L=3, d=2]
   K = [[[1, 1], [-1, 1], [-1, 1]], [[0, 1], [-1, -1], [-1, -1]]]
   V = [[[1, 1], [-1, 1], [-1, 1]], [[1, 1], [1, -1], [1, -1]]]

3) causal mask（第 i 行 = 第 i 个 token 能看到哪些位置）
   pos 0: [1, 0, 0]
   pos 1: [1, 1, 0]
   pos 2: [1, 1, 1]

4) 最后一个 token（pos 2）在 4 个 Q head 上的注意力
   q0(kv0)  scores=[-1.414, 0.0, 0.0]  weights=[0.108, 0.446, 0.446]  out=[-0.783, 1.0]
   q1(kv0)  scores=[0.0, 1.414, 1.414]  weights=[0.108, 0.446, 0.446]  out=[-0.783, 1.0]
   q2(kv1)  scores=[-0.707, 0.0, 0.0]  weights=[0.198, 0.401, 0.401]  out=[1.0, -0.604]
   q3(kv1)  scores=[0.707, -1.414, -1.414]  weights=[0.807, 0.097, 0.097]  out=[1.0, 0.613]

5) 拼回 8 维 → lm_head → logits → argmax
   merged = [-0.783, 1.0,

### 3.2.5 decode：一步只喂 1 个 token

先回看上一节第 4 步的表：`q0` 和 `q1` 的 `scores` 是 `[-1.414, 0, 0]` 与 `[0, 1.414, 1.414]`，但 `weights` 完全相同。这不是 bug —— 两组 scores 处处相差同一个常数（+1.414），而 **softmax 对整体平移不敏感**。

于是得到一个重要结论：**「4 个 q 互不相同」只是必要条件，不是充分条件** —— q 不同不保证注意力不同。本例里 4 个 head 只产生了 3 种输出。共享 K/V 不会让 4 个头塌成 1 个，但也不保证 4 种都不一样。

接下来是 decode。每步只做四件事：

1. 取上一步 argmax 出来的 token，查表得到 `h [1, 8]`；
2. 算它的 `q [4,1,2]`、`k/v [2,1,2]`，把 k/v **追加**到 cache；
3. 用新的 `q` 对**全部** `L` 个历史 K 做点积 —— 历史 K/V 直接从 cache 读，不重算；
4. `w·V` 得到 4 个 head 的输出，拼回 8 维，过 `lm_head`，argmax 出下一个 token。

两个容易忽略的点：

- **decode 阶段的 causal mask 是"免费"的**：未来的 token 还没被写进 cache，`L` 天然就是"能看到的全部"，不需要再造一个下三角。
- **attention 部分的计算量随 `L` 线性增长**：QKᵀ 要扫 `L` 个历史 token，所以 decode 每步都比上一步贵一点；第 6 章会把这份账算成算术强度。


In [10]:
gen_cache = ContiguousKVCache(cfg, capacity=8, num_layers=1)
with torch.no_grad():
    tr = dec(prompt, gen_cache, trace=True)
nxt = int(tr["logits"][-1].argmax())
generated = [nxt]
print(f"prompt = {prompt.tolist()}：prefill 后 L = {gen_cache.lengths[0]}，第一个生成的 token = {nxt}")
print("之后每一步只输入 1 个 token：只算 1 个 q（[4,1,2]）和 1 对 k/v（[2,1,2]）")

for step in range(4):
    inp, pos = nxt, gen_cache.lengths[0]
    with torch.no_grad():
        tr = dec(torch.tensor([inp]), gen_cache, trace=True)
    top2 = torch.topk(tr["logits"][-1], 2)
    nxt = int(top2.indices[0])
    generated.append(nxt)
    print(f"\nstep {step + 1}：pos = {pos}，输入 token {inp}，cache {pos} → {gen_cache.lengths[0]}")
    for hq in range(HQ):
        w = [round(x, 3) for x in tr["weights"][hq, 0].tolist()]
        print(f"     q{hq}(kv{hq // cfg.group_size})  weights = {w}")
    print(f"     top2: {top2.indices[0].item()}({top2.values[0]:.4f}) / "
          f"{top2.indices[1].item()}({top2.values[1]:.4f})  → 下一个 token = {nxt}")
    print(f"     cache：{gen_cache.lengths[0]} token × 16 B = {gen_cache.nbytes()} B")

print("\n序列：", prompt.tolist() + generated[:-1], "→ 预测", generated[-1])

prompt = [0, 1, 2]：prefill 后 L = 3，第一个生成的 token = 13
之后每一步只输入 1 个 token：只算 1 个 q（[4,1,2]）和 1 对 k/v（[2,1,2]）

step 1：pos = 3，输入 token 13，cache 3 → 4
     q0(kv0)  weights = [0.578, 0.141, 0.141, 0.141]
     q1(kv0)  weights = [0.179, 0.043, 0.043, 0.735]
     q2(kv1)  weights = [0.249, 0.123, 0.123, 0.505]
     q3(kv1)  weights = [0.051, 0.423, 0.423, 0.103]
     top2: 12(0.2395) / 14(0.1886)  → 下一个 token = 12
     cache：4 token × 16 B = 64 B

step 2：pos = 4，输入 token 12，cache 4 → 5
     q0(kv0)  weights = [0.152, 0.037, 0.037, 0.623, 0.152]
     q1(kv0)  weights = [0.096, 0.393, 0.393, 0.023, 0.096]
     q2(kv1)  weights = [0.266, 0.032, 0.032, 0.131, 0.539]
     q3(kv1)  weights = [0.049, 0.413, 0.413, 0.1, 0.024]
     top2: 15(0.3785) / 12(0.2771)  → 下一个 token = 15
     cache：5 token × 16 B = 80 B

step 3：pos = 5，输入 token 15，cache 5 → 6
     q0(kv0)  weights = [0.336, 0.082, 0.082, 0.082, 0.336, 0.082]
     q1(kv0)  weights = [0.093, 0.023, 0.023, 0.384, 0.093, 0.384]
     q2(kv1)  we

### 3.2.6 账本：cache 到底占多少字节

KV cache 的字节只和「层数 × H_kv × d × 长度」有关，**和 H_q 无关**：

$$ \text{bytes} = 2 \times L_{\text{layer}} \times H_{kv} \times d \times L \times \text{bytes}_{\text{dtype}} $$

本节只有 1 层，所以是文档口径（2 层、32 B）的一半：**16 B/token**。

最后做一次等价性检查：一次 prefill 整段，和「前 3 个 + 逐个 decode」，每个位置的 logits 必须相同。这是检验 KV cache 有没有写错的最有效手段（第 5 章还会用完整模型再验一次）。


In [11]:
one_layer = 2 * HKV * D * cfg.dtype_bytes      # 16 B
print(f"每 token：2(K,V) × H_kv({HKV}) × d({D}) × {cfg.dtype_bytes} B = {one_layer} B / 层")
print(f"文档 2 层口径：{one_layer} × 2 = {one_layer * 2} B = cfg.kv_bytes_per_token ✓")
assert one_layer * 2 == cfg.kv_bytes_per_token

print(f"\n{'L':>4} | {'1 层已用(B)':>10} | {'2 层已用(B)':>10} | {'buffer 预留(B)':>13}")
print("-" * 52)
for L in [1, 2, 3, 4, 6, 8]:
    print(f"{L:>4} | {L * one_layer:>10} | {L * cfg.kv_bytes_per_token:>10} | {gen_cache.reserved_bytes():>13}")

full = torch.tensor(prompt.tolist() + generated[:-1])
full_cache = ContiguousKVCache(cfg, capacity=16, num_layers=1)
with torch.no_grad():
    logits_full = dec(full, full_cache)["logits"]

inc_cache = ContiguousKVCache(cfg, capacity=16, num_layers=1)
with torch.no_grad():
    parts = [dec(full[:3], inc_cache)["logits"]]
    for t in range(3, len(full)):
        parts.append(dec(full[t:t + 1], inc_cache)["logits"])
logits_inc = torch.cat(parts, dim=0)

diff = (logits_full - logits_inc).abs().max().item()
print(f"\n一次 prefill {len(full)} 个 token  vs  前 3 个 + 逐 token decode：max|diff| = {diff:.2e}")
print("每个位置的 argmax：", logits_full.argmax(-1).tolist(), "vs", logits_inc.argmax(-1).tolist())
assert diff < 1e-6
assert torch.equal(logits_full.argmax(-1), logits_inc.argmax(-1))
print("→ KV cache 只省算力，不改变结果 ✓")

每 token：2(K,V) × H_kv(2) × d(2) × 2 B = 16 B / 层
文档 2 层口径：16 × 2 = 32 B = cfg.kv_bytes_per_token ✓

   L |   1 层已用(B) |   2 层已用(B) |  buffer 预留(B)
----------------------------------------------------
   1 |         16 |         32 |           128
   2 |         32 |         64 |           128
   3 |         48 |         96 |           128
   4 |         64 |        128 |           128
   6 |         96 |        192 |           128
   8 |        128 |        256 |           128

一次 prefill 7 个 token  vs  前 3 个 + 逐 token decode：max|diff| = 5.96e-08
每个位置的 argmax： [14, 13, 13, 12, 15, 12, 15] vs [14, 13, 13, 12, 15, 12, 15]
→ KV cache 只省算力，不改变结果 ✓


### 3.2.7 这一节要带走的三句话

1. **分头是纯形状操作**：`q` 切成 4 份、`k/v` 切成 2 份，KV head 再被 `repeat_interleave` 复制给同一组的 Q head。
2. **prefill 和 decode 走的是同一个 `forward`**：区别只是 `T = 3` 还是 `T = 1`，以及 cache 里已经有多少历史。
3. **cache 只存 K/V**：省掉的是「把历史 token 重算一遍」的算力，代价是显存随 `L` 线性增长 —— 第 4 节就来算这份显存的账。


## 4. 第 3 章：KV cache 的物理存储与分页

先验证字节账，再实现一个**真正按 block 分配**的 paged KV cache。

文档的对照数字（`max_seq_len = 8`，实际 `L = 3`）：

| | 朴素预留 | 分页（block=4） |
|---|---|---|
| 每序列占用 | `8 × 32 = 256 B` | `1 block = 128 B` |
| 可并发数 | `1024/256 = 4` | `8/1 = 8` |
| 利用率 | `96/256 = 37.5%` | `96/128 = 75%` |

KV cache 存的是 **RoPE 之后的 K** 和原始 V；每层独立存一份。

In [12]:
class PagedKVCache:
    """极简 PagedAttention KV cache：池子按固定大小 block 分配，BF16 存储。"""

    def __init__(self, cfg: TinyToyConfig, dtype: torch.dtype = torch.bfloat16):
        self.cfg = cfg
        self.dtype = dtype
        self.block_bytes = cfg.block_size * cfg.kv_bytes_per_token
        self.num_blocks = cfg.kv_pool_bytes // self.block_bytes
        shape = (self.num_blocks, cfg.block_size, cfg.num_key_value_heads, cfg.head_dim)
        # 每层 K、V 各一块池子：[block, block_size, H_kv, d]
        self.k = [torch.zeros(shape, dtype=dtype) for _ in range(cfg.num_hidden_layers)]
        self.v = [torch.zeros(shape, dtype=dtype) for _ in range(cfg.num_hidden_layers)]
        self.block_tables: Dict[int, List[int]] = {}
        self.lengths: Dict[int, int] = {}
        self.free_blocks = list(range(self.num_blocks))

    def reserve(self, seq_id: int, total_len: int) -> None:
        """保证序列有足够的 block；只有新 block 才从空闲池里拿。"""
        needed = math.ceil(total_len / self.cfg.block_size)
        table = self.block_tables.setdefault(seq_id, [])
        while len(table) < needed:
            if not self.free_blocks:
                raise RuntimeError(f"KV 池耗尽：seq {seq_id} 要第 {len(table) + 1} 个 block")
            table.append(self.free_blocks.pop(0))
        self.lengths[seq_id] = total_len

    def write(self, seq_id: int, layer_idx: int, pos: int, k: torch.Tensor, v: torch.Tensor) -> None:
        """把一个 token 的 [H_kv, d] K/V 写进 block table 指向的物理块。"""
        block_id, offset = divmod(pos, self.cfg.block_size)
        block = self.block_tables[seq_id][block_id]
        self.k[layer_idx][block, offset] = k.to(self.dtype)
        self.v[layer_idx][block, offset] = v.to(self.dtype)

    def _gather(self, buf, seq_id: int, layer_idx: int, length: int) -> torch.Tensor:
        table = self.block_tables[seq_id]
        full, rem = divmod(length, self.cfg.block_size)
        parts = [buf[layer_idx][table[i]] for i in range(full)]
        if rem:
            parts.append(buf[layer_idx][table[full]][:rem])
        return torch.cat(parts, dim=0)  # [length, H_kv, d]

    def read_k(self, seq_id: int, layer_idx: int, length: int) -> torch.Tensor:
        return self._gather(self.k, seq_id, layer_idx, length).float()

    def read_v(self, seq_id: int, layer_idx: int, length: int) -> torch.Tensor:
        return self._gather(self.v, seq_id, layer_idx, length).float()

    def used_blocks(self, seq_id: int) -> int:
        return len(self.block_tables[seq_id])

    def used_bytes(self, seq_id: int) -> int:
        return self.used_blocks(seq_id) * self.block_bytes


# 把 A、B、C 三个 token 按文档数值写进去
demo_k = torch.tensor(
    [
        [[1.0, 0.0], [1.0, 1.0]],   # A: kv_head0, kv_head1
        [[0.0, 1.0], [1.0, -1.0]],  # B
        [[1.0, 1.0], [0.0, 1.0]],   # C
    ]
)
demo_v = torch.tensor(
    [
        [[1.0, 0.0], [2.0, 0.0]],
        [[0.0, 1.0], [0.0, 2.0]],
        [[1.0, 1.0], [1.0, 1.0]],
    ]
)

cache = PagedKVCache(cfg, dtype=torch.bfloat16)
seq = 0
cache.reserve(seq, 3)
for t in range(3):
    for layer in range(cfg.num_hidden_layers):
        cache.write(seq, layer, t, demo_k[t], demo_v[t])

read_back = cache.read_k(seq, 0, 3).transpose(0, 1)
assert torch.allclose(read_back, K_cache)

print("block table =", cache.block_tables[seq])
print("block 字节 =", cache.block_bytes, "B；已用 =", cache.used_bytes(seq), "B")
print("其中真实 KV =", 3 * cfg.kv_bytes_per_token, "B；block 粒度利用率 =",
      f"{3 * cfg.kv_bytes_per_token / cache.used_bytes(seq):.1%}")

actual = 3 * cfg.kv_bytes_per_token
naive_reserve = 8 * cfg.kv_bytes_per_token
naive_concurrency = cfg.kv_pool_bytes // naive_reserve
paged_concurrency = cfg.kv_pool_bytes // cache.used_bytes(seq)

print()
print(f"朴素预留：每序列 {naive_reserve} B -> 并发 {naive_concurrency}，利用率 {actual / naive_reserve:.1%}")
print(f"分页分配：每序列 {cache.used_bytes(seq)} B -> 并发 {paged_concurrency}，利用率 {actual / cache.used_bytes(seq):.1%}")

assert (naive_concurrency, paged_concurrency) == (4, 8)
assert abs(actual / naive_reserve - 0.375) < 1e-9
assert abs(actual / cache.used_bytes(seq) - 0.75) < 1e-9

block table = [0]
block 字节 = 128 B；已用 = 128 B
其中真实 KV = 96 B；block 粒度利用率 = 75.0%

朴素预留：每序列 256 B -> 并发 4，利用率 37.5%
分页分配：每序列 128 B -> 并发 8，利用率 75.0%


## 5. 第 5 章：完整的 TinyToy 模型

前面是手算片段，现在搭一个**结构完整的 decoder-only transformer**：

```
Embedding
  └─ 2 × [RMSNorm → GQA Attention(RoPE + KV cache) → 残差
          → RMSNorm → SwiGLU MLP → 残差]
  └─ RMSNorm → lm_head
```

与文档一致：`W_K/W_V` 只投影出 `H_kv × d = 4` 维；RoPE 之后写入 KV cache；注意力使用 `1/√d` 缩放。

In [13]:
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.weight * x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)


def apply_rope(x: torch.Tensor, positions: torch.Tensor, base: float = 10000.0) -> torch.Tensor:
    """x: [..., T, d]；positions: [T]。RoPE 直接把位置编码进 K，所以缓存存的是旋转后的 K。"""
    half = x.shape[-1] // 2
    freqs = base ** (-torch.arange(half, dtype=torch.float32) / half)
    angles = positions.float()[:, None] * freqs[None, :]          # [T, half]
    cos, sin = angles.cos()[None, None], angles.sin()[None, None]
    x1, x2 = x[..., :half], x[..., half:]
    return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)


class TinyToyBlock(nn.Module):
    def __init__(self, cfg: TinyToyConfig):
        super().__init__()
        self.cfg = cfg
        h, hq, hkv, d, i = (
            cfg.hidden_size,
            cfg.num_attention_heads,
            cfg.num_key_value_heads,
            cfg.head_dim,
            cfg.intermediate_size,
        )
        self.norm1 = RMSNorm(h)
        self.q_proj = nn.Linear(h, hq * d, bias=False)
        self.k_proj = nn.Linear(h, hkv * d, bias=False)
        self.v_proj = nn.Linear(h, hkv * d, bias=False)
        self.o_proj = nn.Linear(hq * d, h, bias=False)
        self.norm2 = RMSNorm(h)
        self.gate_proj = nn.Linear(h, i, bias=False)
        self.up_proj = nn.Linear(h, i, bias=False)
        self.down_proj = nn.Linear(i, h, bias=False)

    def forward(self, x, cache=None, layer_idx=0, seq_id=0, start_pos=0):
        cfg = self.cfg
        T = x.shape[1]
        normed = self.norm1(x)

        q = self.q_proj(normed).view(1, T, cfg.num_attention_heads, cfg.head_dim).transpose(1, 2)
        k = self.k_proj(normed).view(1, T, cfg.num_key_value_heads, cfg.head_dim).transpose(1, 2)
        v = self.v_proj(normed).view(1, T, cfg.num_key_value_heads, cfg.head_dim).transpose(1, 2)

        positions = torch.arange(start_pos, start_pos + T)
        q = apply_rope(q, positions)
        k = apply_rope(k, positions)

        if cache is not None:
            cache.reserve(seq_id, start_pos + T)
            for t in range(T):
                cache.write(seq_id, layer_idx, start_pos + t, k[0, :, t, :], v[0, :, t, :])
            K = cache.read_k(seq_id, layer_idx, start_pos + T).transpose(0, 1)  # [Hkv, L, d]
            V = cache.read_v(seq_id, layer_idx, start_pos + T).transpose(0, 1)
            past = start_pos
        else:
            K, V = k[0], v[0]
            past = 0

        # GQA：每个 KV head 广播给 group_size 个 Q head
        K = K.repeat_interleave(cfg.group_size, dim=0)  # [Hq, L, d]
        V = V.repeat_interleave(cfg.group_size, dim=0)

        scores = torch.einsum("hqd,hld->hql", q[0], K) / math.sqrt(cfg.head_dim)
        L = K.shape[1]
        q_idx = torch.arange(T)[:, None]
        k_idx = torch.arange(L)[None, :]
        causal = k_idx <= (past + q_idx)
        scores = scores.masked_fill(~causal, float("-inf"))
        attn = torch.softmax(scores, dim=-1)

        out = torch.einsum("hql,hld->hqd", attn, V).transpose(0, 1).reshape(T, cfg.hidden_size)
        x = x + self.o_proj(out)[None]

        normed2 = self.norm2(x)
        x = x + self.down_proj(F.silu(self.gate_proj(normed2)) * self.up_proj(normed2))
        return x


class TinyToyModel(nn.Module):
    def __init__(self, cfg: TinyToyConfig, seed: int = 0):
        super().__init__()
        self.cfg = cfg
        self.embed = nn.Embedding(cfg.vocab_size, cfg.hidden_size)
        self.layers = nn.ModuleList([TinyToyBlock(cfg) for _ in range(cfg.num_hidden_layers)])
        self.norm = RMSNorm(cfg.hidden_size)
        self.lm_head = nn.Linear(cfg.hidden_size, cfg.vocab_size, bias=False)

        gen = torch.Generator().manual_seed(seed)
        for module in self.modules():
            if isinstance(module, RMSNorm):
                nn.init.ones_(module.weight)
            elif isinstance(module, (nn.Linear, nn.Embedding)):
                nn.init.normal_(module.weight, mean=0.0, std=0.02, generator=gen)

    def forward(self, tokens, cache=None, seq_id=0, start_pos=0):
        x = self.embed(tokens)  # [1, T, H]
        for i, layer in enumerate(self.layers):
            x = layer(x, cache, layer_idx=i, seq_id=seq_id, start_pos=start_pos)
        return self.lm_head(self.norm(x))


model = TinyToyModel(cfg)
model.eval()

block_linear_params = sum(p.numel() for p in model.layers.parameters() if p.ndim == 2)
total_params = sum(p.numel() for p in model.parameters())

print(f"transformer block 线性参数 = {block_linear_params}（= N，用于 2N）")
print(f"全模型参数 = {total_params}")
print("差值 = embedding + lm_head + RMSNorm 权重，均不计入 2N")
assert block_linear_params == 1152

transformer block 线性参数 = 1152（= N，用于 2N）
全模型参数 = 1448
差值 = embedding + lm_head + RMSNorm 权重，均不计入 2N


## 6. 第 5 章：验证 prefill 与 decode 数值等价

这是整个 notebook 最有价值的一项检查：

1. **一次性 prefill** 5 个 token，取最后一个位置的 logits；
2. 先把前 4 个 token prefill 进 KV cache，再 **decode** 第 5 个 token；
3. 两者必须完全一致。

如果 causal mask、RoPE 位置、KV 写入顺序有一处写错，这个断言就会失败。再做一次逐 token 增量前向，与整段 prefill 的每个位置对比。

In [14]:
tokens = torch.tensor([[3, 1, 4, 1, 5]])

with torch.no_grad():
    # 1) 整段 prefill
    cache_full = PagedKVCache(cfg, dtype=torch.float32)
    logits_full = model(tokens, cache=cache_full, start_pos=0)

    # 2) 前 4 个 token prefill + 第 5 个 token decode
    cache_dec = PagedKVCache(cfg, dtype=torch.float32)
    _ = model(tokens[:, :4], cache=cache_dec, start_pos=0)
    logits_decode = model(tokens[:, 4:], cache=cache_dec, start_pos=4)

    # 3) 完全逐 token 前向
    cache_inc = PagedKVCache(cfg, dtype=torch.float32)
    step_logits = [
        model(tokens[:, t : t + 1], cache=cache_inc, start_pos=t)
        for t in range(tokens.shape[1])
    ]
    logits_inc = torch.cat(step_logits, dim=1)

diff_decode = (logits_full[:, -1] - logits_decode[:, 0]).abs().max().item()
diff_inc = (logits_full - logits_inc).abs().max().item()

print("prefill 最后一个位置 logits:", [round(x, 4) for x in logits_full[0, -1].tolist()])
print("decode 第 5 个 token logits  :", [round(x, 4) for x in logits_decode[0, 0].tolist()])
print(f"max |diff| = {diff_decode:.2e}")
print(f"逐 token 前向 vs 整段 prefill: max |diff| = {diff_inc:.2e}")
print(f"KV block（5 token）: {cache_full.block_tables[0]}，"
      f"占 {cache_full.used_bytes(0)} B，其中 token 账 {5 * cfg.kv_bytes_per_token} B")

assert diff_decode < 1e-5
assert diff_inc < 1e-5
print("\nprefill ≡ decode ✓  位置编码、causal mask、KV cache 全部一致")

prefill 最后一个位置 logits: [-0.0889, 0.0003, -0.0363, -0.049, 0.0744, -0.0725, -0.046, -0.0051, 0.028, 0.1668, 0.0366, 0.015, 0.0468, -0.0039, -0.0381, 0.014]
decode 第 5 个 token logits  : [-0.0889, 0.0003, -0.0363, -0.049, 0.0744, -0.0725, -0.046, -0.0051, 0.028, 0.1668, 0.0366, 0.015, 0.0468, -0.0039, -0.0381, 0.014]
max |diff| = 0.00e+00
逐 token 前向 vs 整段 prefill: max |diff| = 1.49e-08
KV block（5 token）: [0, 1]，占 256 B，其中 token 账 160 B

prefill ≡ decode ✓  位置编码、causal mask、KV cache 全部一致


## 7. 第 4 章：算术强度与 Roofline

$$ \text{AI} = \frac{\text{FLOPs}}{\text{HBM 字节}} \qquad \text{拐点} = \frac{\text{峰值算力}}{\text{峰值带宽}} $$

H200 FP8：`1979 TFLOPS / 4.8 TB/s = 412 FLOP/B`。对线性层：

$$ \text{AI} \approx \frac{2MKN}{KN} = 2M $$

TinyToy 用 BF16 权重时，`AI_linear = M`（字节翻倍、峰值也减半，拐点同样减半）。

下面先把文档第 4 / 5 章的账目写成函数，再逐项断言。

In [15]:
def decode_step(cfg: TinyToyConfig, batch: int, length: int) -> dict:
    """decode 一步：B 条序列，每条已有 L 个 token（含本步新写入的）。"""
    linear = 2 * 576 * cfg.num_hidden_layers * batch
    attn = 4 * length * cfg.head_dim * cfg.num_attention_heads * cfg.num_hidden_layers * batch
    flops = linear + attn
    weights_bytes = 2 * 576 * cfg.num_hidden_layers            # 全批只读一次
    kv_read = batch * length * cfg.kv_bytes_per_token
    kv_write = batch * cfg.kv_bytes_per_token
    hbm = weights_bytes + kv_read + kv_write
    return {
        "flops": flops,
        "bytes": hbm,
        "ai": flops / hbm,
        "linear_ai": linear / weights_bytes,
        "weights": weights_bytes,
        "kv_read": kv_read,
        "kv_write": kv_write,
    }


def prefill_step(cfg: TinyToyConfig, T: int) -> dict:
    """prefill T 个 token，causal；cache 全 miss。"""
    linear = T * 2 * 576 * cfg.num_hidden_layers
    attn = 4 * cfg.head_dim * cfg.num_attention_heads * (T * (T + 1) // 2) * cfg.num_hidden_layers
    flops = linear + attn
    weights_bytes = 2 * 576 * cfg.num_hidden_layers
    kv_write = T * cfg.kv_bytes_per_token
    hbm = weights_bytes + kv_write
    return {
        "flops": flops,
        "bytes": hbm,
        "ai": flops / hbm,
        "linear_ai": linear / weights_bytes,
    }


print(f"{'场景':<26}{'FLOPs':>10}{'字节':>10}{'整体 AI':>10}{'线性 AI':>10}")
print("-" * 68)
scenarios = [
    ("decode B=2, L=3", decode_step(cfg, batch=2, length=3)),
    ("prefill T=3", prefill_step(cfg, T=3)),
    ("prefill T=32", prefill_step(cfg, T=32)),
]
for name, row in scenarios:
    print(f"{name:<26}{row['flops']:>10}{row['bytes']:>10}{row['ai']:>10.2f}{row['linear_ai']:>10.2f}")

d = decode_step(cfg, batch=2, length=3)
p3 = prefill_step(cfg, T=3)
p32 = prefill_step(cfg, T=32)

assert d["flops"] == 4992 and d["bytes"] == 2560 and abs(d["ai"] - 1.95) < 0.01
assert d["weights"] == 2304 and d["kv_read"] == 192 and d["kv_write"] == 64
assert p3["flops"] == 7296 and p3["bytes"] == 2400 and abs(p3["ai"] - 3.04) < 0.01
assert p32["flops"] == 107520 and p32["bytes"] == 3328 and abs(p32["ai"] - 32.3) < 0.05
assert abs(d["linear_ai"] - 2.0) < 1e-9
assert abs(p32["linear_ai"] - 32.0) < 1e-9

print()
print("prefill 的 AI 随 T 近似线性增长（M = 本次 GEMM 的 token 数）：")
for T in [1, 2, 4, 8, 16, 32, 64, 128]:
    row = prefill_step(cfg, T=T)
    print(f"  T={T:<4} AI={row['ai']:6.2f}   （线性部分 AI={row['linear_ai']:.1f}）")

H200 = {"peak_tflops": 1979.0, "bw_tbs": 4.8, "mfu": 0.4, "bw_eff": 0.8}
ridge = H200["peak_tflops"] * 1e12 / (H200["bw_tbs"] * 1e12)
print()
print(f"H200 FP8 拐点 = {H200['peak_tflops']}/{H200['bw_tbs']} = {ridge:.0f} FLOP/B")
print(f"线性层 AI ≈ 2M 时，M 阈值 = {ridge / 2:.0f}（文档取整 206）")
assert abs(ridge - 412.29) < 0.1

场景                             FLOPs        字节     整体 AI     线性 AI
--------------------------------------------------------------------
decode B=2, L=3                 4992      2560      1.95      2.00
prefill T=3                     7296      2400      3.04      3.00
prefill T=32                  107520      3328     32.31     32.00

prefill 的 AI 随 T 近似线性增长（M = 本次 GEMM 的 token 数）：
  T=1    AI=  1.01   （线性部分 AI=1.0）
  T=2    AI=  2.03   （线性部分 AI=2.0）
  T=4    AI=  4.05   （线性部分 AI=4.0）
  T=8    AI=  8.10   （线性部分 AI=8.0）
  T=16   AI= 16.18   （线性部分 AI=16.0）
  T=32   AI= 32.31   （线性部分 AI=32.0）
  T=64   AI= 64.47   （线性部分 AI=64.0）
  T=128  AI=128.64   （线性部分 AI=128.0）

H200 FP8 拐点 = 1979.0/4.8 = 412 FLOP/B
线性层 AI ≈ 2M 时，M 阈值 = 206（文档取整 206）


### 7.1 在本机观察「M 越大，GEMM 越吃得饱」

Roofline 的绝对值依赖硬件，但趋势是通用的：**一次 GEMM 里塞进的 token 越多，算术强度越高，计算单元越不容易空转**。

下面在 CPU 上做一个极小的 `[M,1024] × [1024,1024]` 基准。数字只用于观察趋势，不能与 H200 对比。

> 如果你的机器在跑别的任务，耗时会有波动；重复运行取趋势即可。

In [16]:
def bench_matmul(M: int, K: int = 1024, N: int = 1024, iters: int = 5, warmup: int = 2) -> tuple:
    a = torch.randn(M, K)
    b = torch.randn(K, N)
    for _ in range(warmup):
        torch.mm(a, b)
    start = time.perf_counter()
    for _ in range(iters):
        torch.mm(a, b)
    elapsed = (time.perf_counter() - start) / iters
    flops = 2 * M * K * N
    return elapsed * 1e3, flops / elapsed / 1e9


print(f"{'M':>6}{'单次耗时(ms)':>16}{'实测 GFLOPS':>16}{'AI≈2M':>10}")
print("-" * 50)
results = []
for M in [1, 8, 32, 128, 512, 2048]:
    ms, gflops = bench_matmul(M, iters=3 if M >= 1024 else 5)
    results.append((M, ms, gflops))
    print(f"{M:>6}{ms:>16.3f}{gflops:>16.1f}{2 * M:>10}")

small, large = results[0][2], results[-1][2]
print(f"\nM=2048 的吞吐是 M=1 的 {large / small:.1f} 倍 —— 与 Roofline 的结论同向：")
print("小 M 时权重搬运占主导，计算单元闲置；大 M 才能把权重读取摊薄。")
assert results[-1][2] > results[0][2]

     M        单次耗时(ms)       实测 GFLOPS     AI≈2M
--------------------------------------------------
     1           0.008           261.3         2
     8           0.028           604.9        16
    32           0.046          1458.6        64
   128           0.167          1608.9       256
   512           0.882          1218.1      1024


  2048           2.673          1606.5      4096

M=2048 的吞吐是 M=1 的 6.1 倍 —— 与 Roofline 的结论同向：
小 M 时权重搬运占主导，计算单元闲置；大 M 才能把权重读取摊薄。


## 8. 第 6 章：cached token 的真实成本

请求形状：**8 个 cached + 2 个 uncached + 2 个 output = 12 token**。

文档给出三个完全不同的倍数：

- **算力**：`10/2 = 5×`（cached token 不产生 prefill 计算）
- **并发容量**：`6/2 = 3×`（共享前缀省 block，但不可能无限共享）
- **decode 带宽**：`1×`（KV 每个 decode step 仍要完整读一遍，一分不省）

下面把 block 账算出来并断言。

In [17]:
def cached_cost_multipliers(
    cfg: TinyToyConfig, cached: int = 8, uncached: int = 2, pool_blocks: int | None = None
) -> dict:
    pool_blocks = pool_blocks or cfg.kv_pool_bytes // (cfg.block_size * cfg.kv_bytes_per_token)
    total = cached + uncached

    compute_mult = total / uncached

    # A：共享前缀 + 每请求 1 个私有 block
    shared_blocks = math.ceil(cached / cfg.block_size)
    concurrency_a = pool_blocks - shared_blocks

    # B：没有共享，每个请求独占 ceil(10/4)=3 个 block
    blocks_b = math.ceil(total / cfg.block_size)
    concurrency_b = pool_blocks // blocks_b

    return {
        "compute": compute_mult,
        "concurrency": concurrency_a / concurrency_b,
        "kv_bandwidth": 1.0,  # 每个 decode step 都要读 10 个 token 的 KV
        "detail": {
            "total_tokens": total,
            "shared_blocks": shared_blocks,
            "concurrency_a": concurrency_a,
            "blocks_b": blocks_b,
            "concurrency_b": concurrency_b,
            "decode_kv_read_B": total * cfg.kv_bytes_per_token,
        },
    }


cost = cached_cost_multipliers(cfg)
print(f"算力倍数     = {cost['detail']['total_tokens']}/{2} = {cost['compute']:.0f}×")
print(f"并发容量倍数 = {cost['detail']['concurrency_a']}/{cost['detail']['concurrency_b']} = {cost['concurrency']:.0f}×")
print(f"decode 带宽   = {cost['kv_bandwidth']:.0f}×（KV 读取不可摊薄、不可省略）")
print(f"每步每序列 KV 读取 = {cost['detail']['decode_kv_read_B']} B（A、B 相同）")

assert cost["compute"] == 5
assert cost["concurrency"] == 3
assert cost["kv_bandwidth"] == 1

算力倍数     = 10/2 = 5×
并发容量倍数 = 6/2 = 3×
decode 带宽   = 1×（KV 读取不可摊薄、不可省略）
每步每序列 KV 读取 = 320 B（A、B 相同）


## 9. 第 7 章：五道闸门与配置死锁

调度器让一个请求从 waiting 进入 running，至少要过三关（另外两关是抢占与队头阻塞）：

| 闸门 | 检查 |
|---|---|
| ① KV block | `num_new_blocks ≤ free_blocks`（还要为下一步增长预留） |
| ② 槽位 | `running < max_num_seqs` |
| ③ token 预算 | `used + chunk ≤ max_num_batched_tokens` |

文档 A.12 的三个例子（`seqs=4`，`budget=8`，池 8 block）：

- 4 条 decode：槽位 100%、预算 50%、KV 50%
- 1 个 12-token prompt：预算 150%，**进不来**
- 1 个 8-token chunk：预算 100%，能进

再演示 7.11 的配置死锁：当 `max_num_seqs == max_num_batched_tokens` 时，decode 吃光预算，prefill 额度恒为 0。

In [18]:
def admission_gates(
    cfg: TinyToyConfig, running: int, used_tokens: int, chunk_tokens: int, free_blocks: int, new_blocks: int
) -> dict:
    gates = {
        "① KV block": new_blocks <= free_blocks,
        "② 槽位": running < cfg.max_num_seqs,
        "③ token 预算": used_tokens + chunk_tokens <= cfg.max_num_batched_tokens,
    }
    return {"gates": gates, "admit": all(gates.values())}


cases = [
    ("再来 1 条 decode", dict(running=4, used_tokens=4, chunk_tokens=1, free_blocks=4, new_blocks=1)),
    ("12-token prompt", dict(running=2, used_tokens=0, chunk_tokens=12, free_blocks=4, new_blocks=3)),
    ("8-token chunk", dict(running=2, used_tokens=0, chunk_tokens=8, free_blocks=4, new_blocks=2)),
]

for name, kwargs in cases:
    result = admission_gates(cfg, **kwargs)
    mark = "通过" if result["admit"] else "拒绝"
    detail = ", ".join(f"{k}={'✓' if v else '✗'}" for k, v in result["gates"].items())
    print(f"{name:<18} -> {mark:<3} [{detail}]")

assert admission_gates(cfg, **cases[0][1])["admit"] is False   # 槽位满（场景 D）
assert admission_gates(cfg, **cases[1][1])["admit"] is False   # 预算 12 > 8
assert admission_gates(cfg, **cases[2][1])["admit"] is True    # 刚好 8/8


def prefill_headroom(cfg: TinyToyConfig, running: int) -> int:
    """留给 prefill 的 token 额度；<=0 时新请求永远进不来。"""
    return cfg.max_num_batched_tokens - running


bad = TinyToyConfig(max_num_seqs=8, max_num_batched_tokens=8)
print()
print(f"正常配置：4 条 decode 时 prefill 额度 = {prefill_headroom(cfg, 4)} token")
print(f"死锁配置：8 条 decode 时 prefill 额度 = {prefill_headroom(bad, 8)} token")
print("→ 槽位 100%、预算 100%，但 prefill 永远排不进来；这是配置错误，不是容量不足。")

assert prefill_headroom(cfg, 4) == 4
assert prefill_headroom(bad, 8) == 0
assert bad.max_num_seqs <= bad.max_num_batched_tokens

再来 1 条 decode      -> 拒绝  [① KV block=✓, ② 槽位=✗, ③ token 预算=✓]
12-token prompt    -> 拒绝  [① KV block=✓, ② 槽位=✓, ③ token 预算=✗]
8-token chunk      -> 通过  [① KV block=✓, ② 槽位=✓, ③ token 预算=✓]

正常配置：4 条 decode 时 prefill 额度 = 4 token
死锁配置：8 条 decode 时 prefill 额度 = 0 token
→ 槽位 100%、预算 100%，但 prefill 永远排不进来；这是配置错误，不是容量不足。


## 10. 第 8 章：四维水位

水位必须**分维度算、取 max、标注瓶颈**：

$$
\begin{aligned}
\text{算力水位} &= \frac{(\text{uncached\_input\_tps} + \text{output\_tps}) \times 2N}{\text{峰值算力} \times \text{MFU}} \\
\text{带宽水位} &= \frac{\text{权重字节} + \text{并发} \times \text{ctx} \times \text{KV/token}}{\text{HBM 带宽} \times \text{效率}} \\
\text{显存水位} &= \frac{\text{活跃 KV 字节}}{\text{KV 池字节}} \\
\text{槽位水位} &= \frac{\text{running}}{\text{max\_num\_seqs}}
\end{aligned}
$$

假想玩具 GPU（文档 A.13）：每步带宽预算 4096 B，拐点 4 FLOP/B → 算力预算 16384 FLOP，KV 池 1024 B，槽位 4。

两个场景的预期结果：

| 场景 | 算力 | 带宽 | 显存 | 槽位 | 系统 |
|---|---|---|---|---|---|
| B=2, L=3 | 30.5% | **62.5%** | 18.75% | 50% | **带宽** |
| B=2, L=15 | 39.8% | 81.25% | **100%（block）** | 50% | **KV** |

In [19]:
BUDGETS = {
    "flops": 4096 * 4,       # 每步算力预算
    "bytes": 4096,           # 每步带宽预算
    "kv": cfg.kv_pool_bytes,
    "seqs": cfg.max_num_seqs,
}
RIDGE = BUDGETS["flops"] / BUDGETS["bytes"]


def water_levels(flops: int, hbm_bytes: int, kv_bytes: int, num_seqs: int) -> dict:
    levels = {
        "算力": flops / BUDGETS["flops"],
        "带宽": hbm_bytes / BUDGETS["bytes"],
        "显存": kv_bytes / BUDGETS["kv"],
        "槽位": num_seqs / BUDGETS["seqs"],
    }
    bottleneck = max(levels, key=levels.get)
    return {"levels": levels, "bottleneck": bottleneck, "system": levels[bottleneck]}


def kv_block_bytes(length: int) -> int:
    """按 block 粒度计算一条序列的 KV 占用。"""
    return math.ceil(length / cfg.block_size) * cfg.block_size * cfg.kv_bytes_per_token


print(f"玩具 GPU：算力预算 {BUDGETS['flops']} FLOP/步，带宽预算 {BUDGETS['bytes']} B/步，拐点 {RIDGE:.0f} FLOP/B")
print(f"KV 池 {BUDGETS['kv']} B，槽位 {BUDGETS['seqs']}\n")

scenarios = []
for L, use_block in [(3, False), (15, True)]:
    row = decode_step(cfg, batch=2, length=L)
    kv_token = 2 * L * cfg.kv_bytes_per_token
    kv_used = 2 * kv_block_bytes(L) if use_block else kv_token
    levels = water_levels(row["flops"], row["bytes"], kv_used, num_seqs=2)
    scenarios.append((L, row, kv_token, kv_used, levels))

print(f"{'场景':<12}{'算力':>8}{'带宽':>8}{'显存':>8}{'槽位':>8}{'系统':>10}{'瓶颈':>8}")
print("-" * 64)
for L, row, kv_token, kv_used, levels in scenarios:
    lv = levels["levels"]
    print(
        f"B=2,L={L:<7}{lv['算力']:>7.1%}{lv['带宽']:>8.1%}{lv['显存']:>8.1%}"
        f"{lv['槽位']:>8.1%}{levels['system']:>10.1%}{levels['bottleneck']:>8}"
    )

L3 = scenarios[0][4]
L15 = scenarios[1][4]
assert abs(L3["levels"]["算力"] - 0.305) < 0.01
assert abs(L3["levels"]["带宽"] - 0.625) < 1e-9
assert abs(L3["levels"]["显存"] - 0.1875) < 1e-9
assert L3["bottleneck"] == "带宽"

assert abs(L15["levels"]["算力"] - 0.398) < 0.01
assert abs(L15["levels"]["带宽"] - 0.8125) < 1e-9
assert L15["levels"]["显存"] == 1.0
assert L15["bottleneck"] == "显存"

print()
print("L=15 的 block 粒度效应：")
print(f"  token 账 = 2 × 15 × 32 = {2 * 15 * cfg.kv_bytes_per_token} B -> {2 * 15 * cfg.kv_bytes_per_token / 1024:.2%}")
print(f"  block 账 = 2 × {math.ceil(15 / cfg.block_size)} block × {cfg.block_size * cfg.kv_bytes_per_token} B = {2 * kv_block_bytes(15)} B -> 100.00%")
print("  → 实际水位比 token 账更高，一个 block 都不剩，新请求进不来。")

# 自洽性：算力水位 / 带宽水位 必须等于 AI / 拐点
ai = decode_step(cfg, batch=2, length=3)["ai"]
ratio_levels = L3["levels"]["算力"] / L3["levels"]["带宽"]
ratio_roofline = ai / RIDGE
print()
print(f"自洽检查：算力/带宽 = {ratio_levels:.3f}，AI/拐点 = {ai:.2f}/{RIDGE:.0f} = {ratio_roofline:.3f}")
assert abs(ratio_levels - ratio_roofline) < 0.001

玩具 GPU：算力预算 16384 FLOP/步，带宽预算 4096 B/步，拐点 4 FLOP/B
KV 池 1024 B，槽位 4

场景                算力      带宽      显存      槽位        系统      瓶颈
----------------------------------------------------------------
B=2,L=3        30.5%   62.5%   18.8%   50.0%     62.5%      带宽
B=2,L=15       39.8%   81.2%  100.0%   50.0%    100.0%      显存

L=15 的 block 粒度效应：
  token 账 = 2 × 15 × 32 = 960 B -> 93.75%
  block 账 = 2 × 4 block × 128 B = 1024 B -> 100.00%
  → 实际水位比 token 账更高，一个 block 都不剩，新请求进不来。

自洽检查：算力/带宽 = 0.487，AI/拐点 = 1.95/4 = 0.487


## 11. 小结：文档结论与代码验证对照

| # | 文档结论 | notebook 验证 |
|---|---|---|
| 1 | `head_dim = 8/4 = 2`，4 个 Q 头互不相同 | `q_heads` 4 个向量各不相同 |
| 2 | 同一份 KV、不同 q → 不同注意力分布 | `out0 = [0.845,0.578]`，`out1 = [0.578,0.845]` |
| 3 | `2N = 2304 FLOPs/token` | 用 `nn.Linear` 统计参数并断言 |
| 4 | `KV = 32 B/token`，池 1024 B = 8 block | `kv_bytes_per_token`、`num_blocks` |
| 5 | 分页把并发从 4 提到 8，利用率 37.5% → 75% | `PagedKVCache` 真实写入并断言 |
| 6 | decode n 步与整段 prefill 数值一致 | `max abs(diff) < 1e-5` |
| 7 | decode AI 1.95、prefill T=3 AI 3.04、T=32 AI 32.3 | 三条断言 |
| 8 | H200 FP8 拐点 412，线性层 M 阈值 206 | 公式计算并断言 |
| 9 | cached token：算力 5× / 并发 3× / 带宽 1× | `cached_cost_multipliers` |
| 10 | 槽位满、预算超、配置死锁是三种不同拒绝 | `admission_gates` + `prefill_headroom` |
| 11 | 四维水位取 max 并标注瓶颈 | 62.5%（带宽）与 100%（KV） |
| 12 | 算力/带宽水位之比 = AI/拐点 | 0.488 vs 0.4875 |
| 13 | 共享 K/V 不塌缩，但「q 不同」只是必要条件 | 4 个头在 prefill 末尾只出 3 种输出；单 token 时恒等于 `v` |

**可以继续做的实验**（留作练习）：

1. 把 `cfg.num_key_value_heads` 改成 4（MHA）或 1（MQA），观察 KV 字节与 AI 的变化，但 attention FLOPs 不变。
2. 把 `cfg.dtype_bytes` 改成 1（FP8 KV），重算第 8 章水位。
3. 在 `PagedKVCache` 上加 `free(seq_id)`，模拟一个请求结束后 block 回收，观察并发恢复。
4. 在 3.2.5 的 greedy 循环上加 temperature / top-k 采样，对比输出序列。